# Perch 2.0 on the BirdSet test sets

Evaluation of the released Perch 2.0 model ([Kaggle: `google/bird-vocalization-classifier`, `tensorFlow2/perch_v2/2`](https://www.kaggle.com/models/google/bird-vocalization-classifier)) on all eight BirdSet test sets (HSN, NBP, NES, PER, POW, SNE, SSW, UHH). These are the values behind the Perch 2.0 row of the BirdSet comparison table in the paper (mean AUROC 0.908, cmAP 0.445, T1-Acc 0.698).

* **Data:** the `test_5s` split of each dataset from the Hugging Face dataset `DBD-research-group/BirdSet`; every 5 s window is zero-padded to 5 s at 32 kHz.
* **Label space:** the model's logits are restricted to the species of the respective dataset, mapped via the eBird codes shipped with the model.
* **Metrics:** AUROC and cmAP are the means of the per-class ROC-AUC and average precision over all classes with at least one positive window (sigmoid of the logits); T1-Acc is BirdSet's top-1 accuracy, which excludes windows without a call.
* All eight datasets are evaluated in one loop below; each prints its own results, and the last cell summarises them.

Requirements: `tensorflow[and-cuda]~=2.20`, `tensorflow_hub`, `datasets==3.6.0`, `torchmetrics`, `torch`, `librosa`, `scikit-learn`, `pandas`. The model runs on a CUDA GPU.

In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # keep TensorFlow/XLA start-up logging out of the outputs
warnings.filterwarnings("ignore")
import sys, platform
import numpy as np, pandas as pd, librosa, torch
import tensorflow as tf, tensorflow_hub as hub, datasets
from datasets import load_dataset
from tqdm import tqdm

# let TensorFlow allocate GPU memory on demand instead of reserving the whole card
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
tf.experimental.numpy.experimental_enable_numpy_behavior()

print("python", platform.python_version(), "| tensorflow", tf.__version__, "| tensorflow_hub", hub.__version__,
      "| datasets", datasets.__version__, "| torch", torch.__version__, "| librosa", librosa.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

python 3.12.3 | tensorflow 2.20.0 | tensorflow_hub 0.16.1 | datasets 3.6.0 | torch 2.9.0+cu128 | librosa 0.11.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
MODEL = "https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2"
model = hub.load(MODEL)

# eBird codes of the model's output classes
labels = pd.read_csv(hub.resolve(MODEL) + "/assets/perch_v2_ebird_classes.csv").ebird2021.tolist()

# sanity check on 5 s of noise: embedding, spatial embedding, logits
out = model.signatures["serving_default"](inputs=np.random.uniform(-1, 1, 5 * 32000).astype(np.float32)[np.newaxis, :])
print(out["embedding"].shape, out["spatial_embedding"].shape, out["label"].shape, "|", len(labels), "classes")

I0000 00:00:1794907907.802514 2341596 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 16444 MB memory:  -> device: 0, name: NVIDIA RTX A5000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1794907916.171777 2341864 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


(1, 1536) (1, 16, 4, 1536) (1, 14795) | 14795 classes


## Metrics (BirdSet definitions, as in the original notebook)

In [3]:
from torchmetrics import Metric
import torch

class TopKAccuracy(Metric):
    def __init__(self, topk=1, include_nocalls=False, threshold=0.5, **kwargs):
        super().__init__(**kwargs)
        self.topk = topk
        self.include_nocalls = include_nocalls
        self.threshold = threshold
        self.add_state("correct", default=torch.tensor(0), dist_reduce_fx="sum")
        self.add_state("total", default=torch.tensor(0), dist_reduce_fx="sum")

    def update(self, preds, targets):
        # Get the top-k predictions
        _, topk_pred_indices = preds.topk(self.topk, dim=1, largest=True, sorted=True)
        targets = targets.to(preds.device)
        no_call_targets = targets.sum(dim=1) == 0

        # consider no_call instances (a threshold is needed here!)
        if self.include_nocalls:
            # check if top-k predictions for all-negative instances are less than threshold
            no_positive_predictions = (
                preds.topk(self.topk, dim=1, largest=True).values < self.threshold
            )
            correct_all_negative = no_call_targets & no_positive_predictions.all(dim=1)

        else:
            # no_calls are removed, set to 0
            correct_all_negative = torch.tensor(0).to(targets.device)

        # convert one-hot encoded targets to class indices for positive cases
        expanded_targets = targets.unsqueeze(1).expand(-1, self.topk, -1)
        correct_positive = expanded_targets.gather(
            2, topk_pred_indices.unsqueeze(-1)
        ).any(dim=1)

        # update correct and total, excluding all-negative instances if specified
        self.correct += correct_positive.sum() + correct_all_negative.sum()
        if not self.include_nocalls:
            self.total += targets.size(0) - no_call_targets.sum()
        else:
            self.total += targets.size(0)

    def compute(self):
        return self.correct.float() / self.total

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score

def calculate_auc(targets, outputs):
    np_targets = targets
    np_outputs = outputs
    num_classes = np_targets.shape[1]
    aucs = []
    aucs2 = []

    probs = 1 / (1 + np.exp(-np_outputs))

    for i in range(num_classes):
        np_targets[:, i] = np.where(np_targets[:, i] > 0.5, 1, 0)

        if np.sum(np_targets[:, i]) > 0:
            class_auc = roc_auc_score(np_targets[:, i], probs[:, i])
            aucs.append(class_auc)
            aucs2.append(class_auc)
        else:
            aucs2.append(-1)
    return np.mean(aucs), aucs2

In [5]:
def calculate_map(targets, outputs):
    np_targets = targets
    np_outputs = outputs
    num_classes = targets.shape[1]
    aps = []
    aps2 = []
    probs = 1 / (1 + np.exp(-np_outputs))
    for i in range(num_classes):
        np_targets[:, i] = np.where(np_targets[:, i] > 0.5, 1, 0)

        if np.sum(np_targets[:, i]) > 0:
            class_ap = average_precision_score(np_targets[:, i], probs[:, i])
            aps.append(class_ap)
            aps2.append(class_ap) 
        else:
            aps2.append(-1)

    #cmap = cmAP(targets.shape[-1])
    #print(aps)
    return np.mean(aps), aps2

## Evaluation of all eight BirdSet test sets

In [6]:
def evaluate(dataset_name):
    """Score every 5 s test window of one BirdSet dataset with Perch 2.0 and compute the metrics."""
    dataset = load_dataset("DBD-research-group/BirdSet", dataset_name, trust_remote_code=True)
    names = dataset["train"].features["ebird_code"].names
    relevant = [labels.index(c) for c in names]          # restrict the logits to this dataset's species
    outputs, targets = [], []
    for item in tqdm(dataset["test_5s"], desc=dataset_name, mininterval=60):
        wav, sr = librosa.load(item["audio"]["path"], sr=32_000, mono=True)
        wav_tmp = np.zeros(32_000 * 5)
        wav_tmp[:len(wav)] = wav
        wav = wav_tmp
        assert len(wav) == 32_000 * 5
        t = np.zeros(len(names))
        for i in item["ebird_code_multilabel"]:
            t[i] = 1.0
        targets.append(t)
        model_outputs = model.signatures["serving_default"](inputs=wav[np.newaxis, :])
        outputs.append(model_outputs["label"][0][relevant])
    outputs, targets = np.stack(outputs), np.stack(targets)
    res = {"species": len(names), "windows": len(targets),
           "auroc": calculate_auc(targets.copy(), outputs)[0],
           "map": calculate_map(targets.copy(), outputs)[0],
           "acc": float(TopKAccuracy()(torch.tensor(outputs), torch.tensor(targets)))}
    print(f"{dataset_name}: {res['species']} species, {res['windows']} windows | "
          f"AUROC {res['auroc']:.4f}  cmAP {res['map']:.4f}  T1-Acc {res['acc']:.4f}", flush=True)
    return res

results = {}
for name in ["HSN", "NBP", "NES", "PER", "POW", "SNE", "SSW", "UHH"]:
    results[name] = evaluate(name)

HSN: 100%|██████████| 12000/12000 [02:12<00:00, 90.25it/s]


HSN: 21 species, 12000 windows | AUROC 0.8980  cmAP 0.5271  T1-Acc 0.6623


NBP: 100%|██████████| 539/539 [00:05<00:00, 94.58it/s]

NBP: 51 species, 539 windows | AUROC 0.9317  cmAP 0.6580  T1-Acc 0.7236



NES: 100%|██████████| 24480/24480 [05:34<00:00, 73.16it/s]


NES: 89 species, 24480 windows | AUROC 0.9522  cmAP 0.4002  T1-Acc 0.5586


... (6 outputs left out)

SSW: 100%|██████████| 205200/205200 [39:19<00:00, 86.96it/s]


SSW: 81 species, 205200 windows | AUROC 0.9757  cmAP 0.4727  T1-Acc 0.7884


UHH: 100%|██████████| 36637/36637 [06:52<00:00, 88.78it/s]


UHH: 27 species, 36637 windows | AUROC 0.9131  cmAP 0.3826  T1-Acc 0.5921


## Results

In [7]:
table = pd.DataFrame(results).T[["species", "windows", "auroc", "map", "acc"]]
table.columns = ["species", "test windows", "AUROC", "cmAP", "T1-Acc"]
table = table.astype({"species": "Int64", "test windows": "Int64", "AUROC": float, "cmAP": float, "T1-Acc": float})
table.loc["mean (8 tasks)"] = [pd.NA, pd.NA, *table[["AUROC", "cmAP", "T1-Acc"]].mean()]
table.round(4)

,species,test windows,AUROC,cmAP,T1-Acc
HSN,21,12000,0.8980,0.5271,0.6623
NBP,51,539,0.9317,0.6580,0.7236
NES,89,24480,0.9522,0.4002,0.5586
PER,132,15120,0.7852,0.2294,0.5334
POW,48,4560,0.9206,0.5410,0.9302
SNE,56,23756,0.8866,0.3497,0.7950
SSW,81,205200,0.9757,0.4727,0.7884
UHH,27,36637,0.9131,0.3826,0.5921
mean (8 tasks),<NA>,<NA>,0.9079,0.4451,0.6979
